# 11 — Kerr Nonlinearity Generates Frequency Combs

**Engineering question**

How does nonlinear interaction turn one pump laser into many new optical frequency modes?

Notebook 07 treated the frequency comb as an addressable mode ladder.  
Notebook 11 explains the physical mechanism that populates that ladder:

```text
pump laser
→ high-Q microresonator
→ large intracavity field
→ χ³ Kerr nonlinear mixing
→ four-wave mixing
→ symmetric sidebands
→ cascaded comb growth
```

This notebook is still an engineering model, not a full nonlinear-dynamics simulation. Its job is to make the mechanism visible and prepare the next step: using these modes as quantum resources.


## Setup

This notebook creates:

```text
figures/
results/csv/
results/json/
results/11_outputs.zip
```

The final cell supports both:

- **Google Colab**: starts a real browser download with `files.download(...)`
- **Jupyter**: displays a clickable `FileLink`


In [ ]:
from pathlib import Path
import json
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIG = ROOT / "figures"
RES = ROOT / "results"
CSV = RES / "csv"
JS = RES / "json"

for p in [FIG, RES, CSV, JS]:
    p.mkdir(parents=True, exist_ok=True)

def savefig(fig, filename, dpi=220):
    path = FIG / filename
    fig.tight_layout()
    fig.savefig(path, dpi=dpi)
    print(f"saved: {path}")
    return path

print("ROOT:", ROOT)
print("FIG:", FIG)
print("RES:", RES)


## 1. Kerr comb generation pipeline

A high-Q microresonator stores light long enough that the intracavity intensity becomes large.

At high intensity, the material response is no longer purely linear. The Kerr effect can be represented by:

\[
n = n_0 + n_2 I
\]

where \(I\) is optical intensity.

The engineering arc is:

```text
optical power
→ stored field
→ nonlinear response
→ frequency mixing
→ comb
```


In [ ]:
labels = [
    "Pump\nlaser",
    "High-Q\nmicroresonator",
    "Large\nintracavity\nfield",
    "χ³ Kerr\nnonlinearity",
    "Four-wave\nmixing",
    "Frequency\ncomb",
]

x = np.arange(len(labels))

fig, ax = plt.subplots(figsize=(13, 3.2))

for xi, label in zip(x, labels):
    rect = Rectangle((xi - 0.42, -0.25), 0.84, 0.5, fill=False, linewidth=2)
    ax.add_patch(rect)
    ax.text(xi, 0, label, ha="center", va="center", fontsize=10.5)

for i in range(len(labels) - 1):
    ax.annotate(
        "",
        xy=(i + 0.58, 0),
        xytext=(i + 0.42, 0),
        arrowprops=dict(arrowstyle="->", linewidth=2),
    )

ax.set_title("Kerr Comb Generation Pipeline", fontsize=16)
ax.set_xlim(-0.7, len(labels) - 0.3)
ax.set_ylim(-0.65, 0.7)
ax.axis("off")

savefig(fig, "11_kerr_pipeline.png")
plt.show()


## 2. Four-wave mixing

The signature process is four-wave mixing.

Two pump photons are converted into two new photons at symmetric frequencies:

\[
2\omega_0 = \omega_{-1} + \omega_{+1}
\]

The equation says energy is conserved. It also explains why the first new frequencies appear as sidebands around the pump.


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.2))
ax.axis("off")

# input pump photons
ax.text(0.22, 0.78, "pump photon", ha="center", fontsize=11)
ax.text(0.22, 0.68, r"$\omega_0$", ha="center", fontsize=24)

ax.text(0.22, 0.52, "+", ha="center", fontsize=22)

ax.text(0.22, 0.38, "pump photon", ha="center", fontsize=11)
ax.text(0.22, 0.28, r"$\omega_0$", ha="center", fontsize=24)

# interaction arrow
ax.annotate(
    "",
    xy=(0.56, 0.50),
    xytext=(0.34, 0.50),
    arrowprops=dict(arrowstyle="->", linewidth=2.5),
)
ax.text(0.45, 0.58, r"$\chi^{(3)}$", ha="center", fontsize=18)

# output sidebands
ax.text(0.76, 0.75, "lower sideband", ha="center", fontsize=11)
ax.text(0.76, 0.65, r"$\omega_{-1}$", ha="center", fontsize=24)

ax.text(0.76, 0.50, "+", ha="center", fontsize=22)

ax.text(0.76, 0.32, "upper sideband", ha="center", fontsize=11)
ax.text(0.76, 0.22, r"$\omega_{+1}$", ha="center", fontsize=24)

ax.text(
    0.5,
    0.05,
    r"Energy conservation:  $2\omega_0 = \omega_{-1} + \omega_{+1}$",
    ha="center",
    fontsize=15,
)

ax.set_title("Four-Wave Mixing Creates Symmetric Sidebands", fontsize=16)

savefig(fig, "11_four_wave_mixing.png")
plt.show()


## 3. Cascaded comb growth

The first sidebands are not the end.

Once sidebands exist, additional mixing processes can populate farther resonator modes.

This is the intuitive cascade:

```text
ω₀
→ ω₋₁, ω₀, ω₊₁
→ ω₋₂, ω₋₁, ω₀, ω₊₁, ω₊₂
→ many modes
```


In [ ]:
stages = [
    [0],
    [-1, 0, 1],
    [-2, -1, 0, 1, 2],
    [-4, -3, -2, -1, 0, 1, 2, 3, 4],
]

fig, ax = plt.subplots(figsize=(11, 5.3))

for row, active_modes in enumerate(stages):
    y = len(stages) - 1 - row
    # inactive resonance lattice
    for n in range(-6, 7):
        ax.plot([n, n], [y - 0.18, y + 0.18], linewidth=1, alpha=0.25)
    # active modes
    for n in active_modes:
        height = 0.45 if n == 0 else 0.34
        ax.plot([n, n], [y - height / 2, y + height / 2], linewidth=3)
        ax.scatter([n], [y + height / 2], s=55)
    ax.text(-7.1, y, f"Stage {row + 1}", va="center", fontsize=11)

for y in [2.5, 1.5, 0.5]:
    ax.annotate("", xy=(0, y - 0.28), xytext=(0, y + 0.28), arrowprops=dict(arrowstyle="->", linewidth=1.8))

ax.set_title("Cascaded Comb Growth from the Pump Mode", fontsize=16)
ax.set_xlabel("Mode index")
ax.set_yticks([])
ax.set_xticks([-6, -3, 0, 3, 6])
ax.set_xticklabels([r"$\omega_0-6\Delta\omega$", r"$\omega_0-3\Delta\omega$", r"$\omega_0$", r"$\omega_0+3\Delta\omega$", r"$\omega_0+6\Delta\omega$"])
ax.set_xlim(-7.6, 6.8)
ax.set_ylim(-0.5, 3.6)
ax.grid(True, axis="x", alpha=0.1)

savefig(fig, "11_comb_growth.png")
plt.show()


## 4. Resonator selection

The nonlinear interaction can create new frequencies.

The resonator determines which frequencies are supported.

An idealized comb mode satisfies:

\[
\omega_n = \omega_0 + n\Delta\omega
\]

where \(\Delta\omega\) is the free spectral range.

So the mechanism is two-part:

```text
Kerr mixing generates frequencies.
The resonator selects the allowed ones.
```


In [ ]:
n = np.arange(-9, 10)

fig, ax = plt.subplots(figsize=(11, 5.2))

# continuous generation band
x_cont = np.linspace(-9, 9, 500)
y_cont = 2.55 + 0.08 * np.sin(4 * x_cont)
ax.plot(x_cont, y_cont, linestyle="--", linewidth=2, alpha=0.65)
ax.text(-9.4, 2.55, "possible generated\nfrequencies", ha="right", va="center", fontsize=11)

# allowed resonances
for ni in n:
    ax.plot([ni, ni], [1.35, 1.95], linewidth=2)
ax.text(-9.4, 1.65, "allowed cavity\nresonances", ha="right", va="center", fontsize=11)

# occupied modes after selection
envelope = np.exp(-(n / 6.5) ** 2)
for ni, h in zip(n, envelope):
    ax.scatter([ni], [0.45], s=70 + 120 * h)
ax.text(-9.4, 0.45, "occupied comb\nmodes", ha="right", va="center", fontsize=11)

ax.annotate("", xy=(0, 2.05), xytext=(0, 2.38), arrowprops=dict(arrowstyle="->", linewidth=1.7))
ax.annotate("", xy=(0, 0.72), xytext=(0, 1.25), arrowprops=dict(arrowstyle="->", linewidth=1.7))

ax.set_title("Nonlinear Generation Meets Resonator Selection", fontsize=16)
ax.set_xlim(-11.2, 9.8)
ax.set_ylim(0, 3.0)
ax.set_xticks([-9, -6, -3, 0, 3, 6, 9])
ax.set_yticks([])
ax.set_xlabel("Frequency / mode index")
ax.grid(True, axis="x", alpha=0.08)

savefig(fig, "11_resonator_selection.png")
plt.show()


## 5. Threshold behavior

Comb generation is not linear in pump power.

Below threshold, the resonator behaves mostly as a linear optical system.

Above threshold, nonlinear mixing can seed sidebands and comb growth.

This figure is conceptual: it shows the engineering role of a pump threshold without modeling a specific device.


In [ ]:
pump = np.linspace(0, 10, 400)
threshold = 3.2
comb_power = np.where(pump < threshold, 0, (pump - threshold) ** 1.55)
comb_power = comb_power / comb_power.max()

fig, ax = plt.subplots(figsize=(7.5, 5.0))

ax.plot(pump, comb_power, linewidth=2.8)
ax.axvline(threshold, linestyle="--", linewidth=1.5)

ax.text(threshold + 0.15, 0.1, "threshold", fontsize=11)
ax.text(1.2, 0.06, "linear\nresponse", ha="center", fontsize=10)
ax.text(7.2, 0.72, "comb\ngrowth", ha="center", fontsize=10)

ax.set_title("Nonlinear Comb Threshold", fontsize=16)
ax.set_xlabel("Pump power")
ax.set_ylabel("Comb power")
ax.grid(True, alpha=0.3)

savefig(fig, "11_threshold.png")
plt.show()


## 6. Engineering summary

Notebook 11 explains why the comb can appear from a single pump.

The key physical ingredients are:

- pump laser,
- high-Q resonator,
- Kerr \(\chi^{(3)}\) nonlinearity,
- four-wave mixing,
- resonator mode selection.


In [ ]:
summary = pd.DataFrame(
    [
        {
            "Physical ingredient": "Pump laser",
            "Engineering role": "Provides optical power",
            "Notebook role": "Input resource",
        },
        {
            "Physical ingredient": "High-Q resonator",
            "Engineering role": "Stores circulating field",
            "Notebook role": "Enhances intensity",
        },
        {
            "Physical ingredient": "Kerr χ³",
            "Engineering role": "Creates nonlinear interaction",
            "Notebook role": "Enables mixing",
        },
        {
            "Physical ingredient": "Four-wave mixing",
            "Engineering role": "Generates sidebands",
            "Notebook role": "First new frequencies",
        },
        {
            "Physical ingredient": "Resonator modes",
            "Engineering role": "Fix mode spacing / FSR",
            "Notebook role": "Selects surviving modes",
        },
        {
            "Physical ingredient": "Frequency comb",
            "Engineering role": "Many addressable optical channels",
            "Notebook role": "Output resource",
        },
    ]
)

fig, ax = plt.subplots(figsize=(12, 3.4))
ax.axis("off")

table = ax.table(
    cellText=summary.values,
    colLabels=summary.columns,
    loc="center",
    cellLoc="center",
)

table.auto_set_font_size(False)
table.set_fontsize(10.5)
table.scale(1.2, 1.7)

for (r, c), cell in table.get_celld().items():
    cell.set_linewidth(1.2)
    if r == 0:
        cell.set_text_props(weight="bold")

ax.set_title("Engineering Summary: Kerr Physics Generates Frequency Resources", fontsize=16, pad=20)

savefig(fig, "11_summary.png")

summary.to_csv(CSV / "11_summary.csv", index=False)
summary.to_json(JS / "11_summary.json", orient="records", indent=2)

plt.show()
summary


## 7. Export and download

This cell packages all outputs and starts a download.

It uses Colab's native downloader when available.  
For local Jupyter, it falls back to a clickable `FileLink`.


In [ ]:
zip_path = RES / "11_outputs.zip"

files_to_zip = [
    FIG / "11_kerr_pipeline.png",
    FIG / "11_four_wave_mixing.png",
    FIG / "11_comb_growth.png",
    FIG / "11_resonator_selection.png",
    FIG / "11_threshold.png",
    FIG / "11_summary.png",
    CSV / "11_summary.csv",
    JS / "11_summary.json",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files_to_zip:
        if file.exists():
            z.write(file, file.relative_to(ROOT))

print(f"Created: {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))

summary


## Takeaways

- A pump laser supplies energy to a high-Q microresonator.
- High intracavity intensity activates Kerr \(\chi^{(3)}\) nonlinear optics.
- Four-wave mixing generates symmetric sidebands while conserving energy.
- Cascaded mixing can populate many resonator modes.
- The resonator selects the allowed frequencies through its fixed free spectral range.

## Next notebook

**13 — Two-Mode Squeezing**

Notebook 11 explains how new frequencies are generated.

Notebook 13 asks how those paired modes become quantum resources:

```text
symmetric sidebands
→ quadrature correlations
→ two-mode squeezing
→ EPR-like frequency-bin resources
```
